In [74]:
import pandas as pd

In [75]:
dfDetails = pd.read_csv("../data/raw/details.csv")
dfStats = pd.read_csv("../data/raw/stats.csv")
dfRecs = pd.read_csv("../data/raw/recommendations.csv")


# Cleaning the data
### We are ignoring the columns that contains an high number of empty values and columns that are not important for the sake of the analysis

In [76]:
dfDetails = dfDetails.drop(columns=["title_japanese",
                                    "url",
                                    "image_url",
                                    "start_date",
                                    "end_date",
                                    "synopsis",
                                    "studios",
                                    "demographics",
                                    "season",
                                    "licensors",
                                    "explicit_genres",
                                    "streaming"],
                           errors = "ignore")
dfDetails.drop_duplicates()
dfDetails.head()

,mal_id,title,type,status,score,scored_by,rank,popularity,members,favorites,genres,themes,source,rating,episodes,year,producers
0,59356,-Socket-,Movie,Finished Airing,NaN,NaN,17086.0,22507,195,0,['Comedy'],[],Original,G - All Ages,1.0,NaN,['Nagoya Zokei University']
1,56036,......,Music,Finished Airing,6.53,503.0,NaN,15004,941,2,"['Horror', 'Supernatural']",['Music'],Original,PG-13 - Teens 13 or older,1.0,NaN,[]
2,2928,.hack//G.U. Returner,OVA,Finished Airing,6.65,9745.0,6366.0,5056,22525,31,"['Adventure', 'Drama', 'Fantasy']",['Video Game'],Game,PG-13 - Teens 13 or older,1.0,NaN,"['Bandai Visual', 'CyberConnect2']"
3,3269,.hack//G.U. Trilogy,Movie,Finished Airing,7.06,15373.0,4194.0,4215,34264,104,"['Action', 'Fantasy']",['Video Game'],Game,PG-13 - Teens 13 or older,1.0,NaN,['Bandai Visual']
4,4469,.hack//G.U. Trilogy: Parody Mode,Special,Finished Airing,6.35,4317.0,8182.0,6696,11135,10,"['Comedy', 'Fantasy', 'Sci-Fi']","['Parody', 'Video Game']",Game,PG-13 - Teens 13 or older,1.0,NaN,['Bandai Visual']


### Same goes for the stats, ignoring columns that can be obtained with simple calculations or obtained in other ways

##### Decided to keep the score_x_votes to analyze better in the future§

In [77]:
dfStats = dfStats.drop(columns=dfStats.filter(regex="percentage$").columns)
dfStats.drop_duplicates()
dfStats.head()

,mal_id,watching,completed,on_hold,dropped,plan_to_watch,total,score_1_votes,score_2_votes,score_3_votes,score_4_votes,score_5_votes,score_6_votes,score_7_votes,score_8_votes,score_9_votes,score_10_votes
0,59356,7,146,4,20,20,197,2.0,0.0,3.0,6.0,25.0,33.0,19.0,2.0,0.0,1.0
1,56036,21,770,8,29,113,941,5.0,6.0,8.0,14.0,50.0,138.0,144.0,81.0,17.0,40.0
2,2928,451,14953,302,349,6472,22527,101.0,93.0,164.0,457.0,1184.0,2054.0,2709.0,1500.0,875.0,608.0
3,3269,726,22790,452,537,9762,34267,120.0,156.0,260.0,560.0,1270.0,2457.0,4157.0,3075.0,1919.0,1400.0
4,4469,241,6918,182,266,3528,11135,83.0,104.0,182.0,292.0,683.0,888.0,871.0,592.0,308.0,315.0


### Cleaning the recommendation by removing duplicates and eventual self recommendation

In [78]:
dfRecs.head()
dfRecs = dfRecs[dfRecs["mal_id"] != dfRecs["recommendation_mal_id"]]
dfRecs.drop_duplicates()

,mal_id,recommendation_mal_id
0,3269,317
1,3269,6922
2,3269,299
3,3269,3446
4,3269,5681
...,...,...
105244,31245,1689
105245,31245,35434
105246,31245,31798
105247,31245,21995


### Decided to merge dfDetails and dfStats to make a complete dataset of the anime

##### Freeing Memory by setting the previous df to None

In [79]:
animeDf = dfDetails.merge(dfStats, how = "left", on = "mal_id")
dfStats = None
dfDetails = None

### Renaming
##### Thinking about changing the name of some columns to make them more clear.
##### Leaving mal_id to make the df more compatible with the others

In [80]:
animeDf = animeDf.rename(columns={"total": "total_users"})
animeDf.head()

,mal_id,title,type,status,score,scored_by,rank,popularity,members,favorites,...,score_1_votes,score_2_votes,score_3_votes,score_4_votes,score_5_votes,score_6_votes,score_7_votes,score_8_votes,score_9_votes,score_10_votes
0,59356,-Socket-,Movie,Finished Airing,NaN,NaN,17086.0,22507,195,0,...,2.0,0.0,3.0,6.0,25.0,33.0,19.0,2.0,0.0,1.0
1,56036,......,Music,Finished Airing,6.53,503.0,NaN,15004,941,2,...,5.0,6.0,8.0,14.0,50.0,138.0,144.0,81.0,17.0,40.0
2,2928,.hack//G.U. Returner,OVA,Finished Airing,6.65,9745.0,6366.0,5056,22525,31,...,101.0,93.0,164.0,457.0,1184.0,2054.0,2709.0,1500.0,875.0,608.0
3,3269,.hack//G.U. Trilogy,Movie,Finished Airing,7.06,15373.0,4194.0,4215,34264,104,...,120.0,156.0,260.0,560.0,1270.0,2457.0,4157.0,3075.0,1919.0,1400.0
4,4469,.hack//G.U. Trilogy: Parody Mode,Special,Finished Airing,6.35,4317.0,8182.0,6696,11135,10,...,83.0,104.0,182.0,292.0,683.0,888.0,871.0,592.0,308.0,315.0


In [81]:
dfRecs = dfRecs.rename(columns={"recommendation_mal_id": "recommended_anime_id"})

### Saving the normalized dfs

In [82]:
animeDf.to_csv("../data/cleaned/anime.csv", index = False)
dfRecs.to_csv("../data/cleaned/recommendation.csv", index = False)

In [83]:
animeDf = None
dfRecs = None